# 10年定着予測 - 探索的データ分析レポート v5（入社時固定・客観条件の深掘り）

**位置づけ**: `data_exploration_v4_report.md`に続く第5弾。1位のPublicスコア(0.50081)に対し、
現在の最良(20_ E_memo単体, Public 0.534829)はまだ約0.034の差がある。ユーザーとの合意に基づき、
今回は「入社時に固定される客観的な条件」（属性・希望・条件の充足度など）に絞って、
E（メモ構造化）・J（希望勤務地マッチ度）に匹敵する新規シグナルを探索する。

検証する候補:
1. 初期部署ID は初期職種を超える追加情報を持つか（v2で「強力な候補」とされたが未検証だった論点）
2. 最終学歴×入社時年齢の整合性（浪人・留年・ブランクの代理指標）
3. 初任給の勤務地内偏差（v4で「弱い有望」止まりだった副産物の本格検証）
4. 前職職種×初期職種のマッチ度（中途入社者、専攻×職種マッチ[v3]の類似分析）
5. キャリア志向×初期役割トラックの整合性
6. **転居許容×希望勤務地マッチの交互作用**（新規探索）


In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")  # Colab想定。ローカル実行時は適宜書き換え
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path(".").resolve()
    while not (PROJECT_ROOT / "data" / "input").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "data" / "input"
TARGET_COL = "10年定着ラベル"
print(PROJECT_ROOT)

/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026


In [2]:
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
y = train_persona[TARGET_COL]
print(f"Train Persona: {train_persona.shape}, Test Persona: {test_persona.shape}")
print(f"定着率: {y.mean():.4f}")

Train Persona: (2761, 20), Test Persona: (2502, 19)
定着率: 0.5647


## 1. 初期部署ID は初期職種を超える追加情報を持つか

`data_exploration_v2_report.md`セクション5は、部署IDの定着率レンジが0%〜82%と大きいことから
「強力な特徴量候補」と評価していた。しかし**部署IDが初期職種の単なる下位区分（部署→職種が1対1）**
だった場合、部署IDのTarget Encodingは実質的に職種の情報を再パッケージしているだけで、
新規の予測力を持たない可能性がある。この点はv2では検証されていなかったため、ここで確認する。

In [3]:
dept_job_nunique = train_persona.groupby("初期部署ID")["初期職種"].nunique()
print("部署あたりの職種数の分布:")
print(dept_job_nunique.value_counts())
print(f"\n全部署が単一職種に従属している: {(dept_job_nunique == 1).all()}")

部署あたりの職種数の分布:
初期職種
1    460
Name: count, dtype: int64

全部署が単一職種に従属している: True


In [4]:
def smoothed_te_within_group_oof(df, col, group_col, target, n_splits=5, smoothing=10, seed=42):
    '''colの値を、所属group_colの平均へスムージングして縮約するOOF Target Encoding'''
    oof = np.full(len(df), np.nan)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        tr = df.iloc[tr_idx].copy()
        tr["_y"] = target.iloc[tr_idx].values
        group_mean = tr.groupby(group_col)["_y"].mean()
        stats_df = tr.groupby(col)["_y"].agg(["mean", "count"])
        col_to_group = tr.drop_duplicates(col).set_index(col)[group_col]
        stats_df["group_mean"] = col_to_group.map(group_mean)
        smoothed = (stats_df["mean"] * stats_df["count"] + stats_df["group_mean"] * smoothing) / (
            stats_df["count"] + smoothing
        )
        val = df.iloc[val_idx]
        val_group_mean = val[group_col].map(group_mean)
        mapped = val[col].map(smoothed)
        oof[val_idx] = mapped.fillna(val_group_mean).values
    return oof


def group_te_oof(df, group_col, target, n_splits=5, seed=42):
    oof = np.full(len(df), np.nan)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        gmean = target.iloc[tr_idx].groupby(df[group_col].iloc[tr_idx]).mean()
        oof[val_idx] = df[group_col].iloc[val_idx].map(gmean).values
    return oof


job_oof = group_te_oof(train_persona, "初期職種", y)
print("初期職種 単体 OOF相関 vs target:", np.corrcoef(job_oof, y)[0, 1])

for sm in [3, 5, 10, 20, 50]:
    dept_te = smoothed_te_within_group_oof(train_persona, "初期部署ID", "初期職種", y, smoothing=sm)
    dev = dept_te - job_oof
    print(f"smoothing={sm:>3}: 部署TE相関={np.corrcoef(dept_te, y)[0,1]:.4f}  職種内偏差の相関={np.corrcoef(dev, y)[0,1]:.4f}")

初期職種 単体 OOF相関 vs target: 0.22446073191313642
smoothing=  3: 部署TE相関=0.1506  職種内偏差の相関=0.0009
smoothing=  5: 部署TE相関=0.1685  職種内偏差の相関=-0.0008
smoothing= 10: 部署TE相関=0.1934  職種内偏差の相関=-0.0030
smoothing= 20: 部署TE相関=0.2113  職種内偏差の相関=-0.0047
smoothing= 50: 部署TE相関=0.2214  職種内偏差の相関=-0.0061


### 考察（1）: 部署IDは職種を超える新規情報を持たない（負の結果）

全460部署が例外なく単一の初期職種に従属しており（部署は職種の下位区分）、部署IDのTarget Encodingは
スムージングを強めるほど職種単体の相関(0.224)に漸近する。**職種平均で正しく縮約した場合の
「部署の職種内偏差」は目的変数とほぼ無相関（相関 -0.003 前後）**であり、部署IDは職種以上の
追加情報を持たない。v2の「強力な候補」という評価は、職種への従属を考慮しないTarget Encodingで
測定していたための見かけの効果だったと考えられる。**この方向は不採用とする。**

## 2. 最終学歴×入社時年齢の整合性

新卒については「入社時年齢 − 学歴に対応する標準卒業年齢」を浪人・留年・入社までのブランクの代理指標、
中途については「(入社時年齢-標準卒業年齢)×12 − 前職経験月数」を未説明ブランク期間の代理指標として検証する。

In [5]:
STD_GRAD_AGE = {"高校卒": 18, "専門学校卒": 20, "短大・高専卒": 20, "大学卒": 22, "大学院卒": 24}
train_persona["std_grad_age"] = train_persona["最終学歴"].map(STD_GRAD_AGE)

new_grad = train_persona[train_persona["入社区分"] == "新卒"].copy()
new_grad["age_gap"] = new_grad["入社時年齢"] - new_grad["std_grad_age"]
print("新卒: 年齢ギャップ vs target 相関:", np.corrcoef(new_grad["age_gap"], new_grad[TARGET_COL])[0, 1])
gap_bins = pd.cut(new_grad["age_gap"], bins=[-1, 0, 1, 2, 100], labels=["0(標準)", "1年遅れ", "2年遅れ", "3年以上遅れ"])
print(new_grad[TARGET_COL].groupby(gap_bins).agg(["mean", "count"]))

mid = train_persona[train_persona["入社区分"] == "中途"].copy()
mid["unexplained_gap_months"] = (mid["入社時年齢"] - mid["std_grad_age"]) * 12 - mid["前職経験月数"]
print("\n中途: 未説明ブランク(月) vs target 相関:", np.corrcoef(mid["unexplained_gap_months"], mid[TARGET_COL])[0, 1])
gap_bins2 = pd.qcut(mid["unexplained_gap_months"], 5, duplicates="drop")
print(mid[TARGET_COL].groupby(gap_bins2).agg(["mean", "count"]))

新卒: 年齢ギャップ vs target 相関: 0.004157591596910328
             mean  count
age_gap                 
0(標準)    0.524638   1035
1年遅れ     0.508323    781
2年遅れ     0.534392    189
3年以上遅れ   0.650000     20

中途: 未説明ブランク(月) vs target 相関: -0.051944906475858206
                            mean  count
unexplained_gap_months                 
(-0.001, 2.0]           0.679739    153
(2.0, 9.0]              0.746575    146
(9.0, 15.0]             0.673333    150
(15.0, 22.0]            0.701389    144
(22.0, 54.0]            0.629371    143


### 考察（2）: 学歴・年齢の整合性は無風（負の結果）

新卒の年齢ギャップは相関0.004とほぼゼロ、ビン別でも単調な傾向がない。中途の未説明ブランクも
相関-0.052と弱く、ビン別の定着率（0.63〜0.75）に単調なパターンは見られない。**この方向は不採用とする。**

## 3. 初任給の勤務地内偏差（v4副産物の本格検証）

v4では相関0.130の「弱い有望」とされていたが、これが初任給そのものの効果と独立か、
既存の等級内偏差と冗長でないかを確認する。

In [6]:
loc_mean_salary_oof = group_te_oof(train_persona, "初期勤務地", train_persona["初任給_円"])
# 上のgroup_te_oofはtarget前提の実装のため、給与用に読み替えて使う（同じロジックで平均を計算しているだけ）
def oof_group_mean(df, col, group_col, n_splits=5, seed=42):
    oof = np.full(len(df), np.nan)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        gmean = df[col].iloc[tr_idx].groupby(df[group_col].iloc[tr_idx]).mean()
        oof[val_idx] = df[group_col].iloc[val_idx].map(gmean).values
    return oof


loc_mean_salary_oof = oof_group_mean(train_persona, "初任給_円", "初期勤務地")
salary_loc_dev = train_persona["初任給_円"] - loc_mean_salary_oof
print("初任給の勤務地内偏差 相関 vs target:", np.corrcoef(salary_loc_dev, y)[0, 1])
print("(参考)初任給そのものの相関 vs target:", np.corrcoef(train_persona["初任給_円"], y)[0, 1])

both_mean_oof = np.full(len(train_persona), np.nan)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, val_idx in kf.split(train_persona):
    g = train_persona[["初期勤務地", "初期等級"]].iloc[tr_idx].apply(tuple, axis=1)
    gmean = train_persona["初任給_円"].iloc[tr_idx].groupby(g).mean()
    global_mean = train_persona["初任給_円"].iloc[tr_idx].mean()
    vg = train_persona[["初期勤務地", "初期等級"]].iloc[val_idx].apply(tuple, axis=1)
    both_mean_oof[val_idx] = vg.map(gmean).fillna(global_mean).values

salary_both_dev = train_persona["初任給_円"] - both_mean_oof
print("勤務地×等級 両方コントロール後の偏差 相関 vs target:", np.corrcoef(salary_both_dev, y)[0, 1])

初任給の勤務地内偏差 相関 vs target: 0.12946005759680176
(参考)初任給そのものの相関 vs target: 0.13158418040308018
勤務地×等級 両方コントロール後の偏差 相関 vs target: -0.022252123442021512


### 考察（3）: 勤務地内偏差は初任給そのものと重複（負の結果）

勤務地内偏差の相関(0.129)は初任給そのものの相関(0.132)とほぼ同じで、独立した追加情報は乏しい。
さらに勤務地×等級の両方をコントロールすると相関はほぼ消える（-0.022）。給与の予測力はほぼ全て
既存の「初任給_円」「初期等級」（すでにモデルに投入済み）で説明され尽くしている。**不採用とする。**

## 4. 前職職種×初期職種のマッチ度（中途入社者）

v3で専攻×職種のマッチ度は非有意だったが、専攻ではなく実際の前職の職種と初期職種が
一致しているかどうかは未検証だった。

In [7]:
mid_job = train_persona[train_persona["前職職種"].notna()].copy()
job_match = mid_job["前職職種"] == mid_job["初期職種"]
print(f"中途入社者 n={len(mid_job)}")
print(mid_job[TARGET_COL].groupby(job_match).agg(["mean", "count"]))
ct = pd.crosstab(job_match, mid_job[TARGET_COL])
chi2, p, dof, exp = chi2_contingency(ct)
print(f"カイ二乗検定 p={p:.4f}")

中途入社者 n=736
           mean  count
False  0.686567    134
True   0.686047    602
カイ二乗検定 p=1.0000


### 考察（4）: 前職職種マッチも無風（負の結果）

定着率はマッチ68.6% vs ミスマッチ68.7%とほぼ完全に一致（p=1.0）。専攻×職種（v3）と同様、
配属は専門性の継続よりも欠員状況等で決まっている可能性が高い。**不採用とする。**

## 5. キャリア志向×初期役割トラックの整合性

「管理職志向・専門職志向」を持つ社員が、実際にメンバーどまりで配属された場合に定着率が下がるかを検証する。

In [8]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


def extract_career_section(text):
    m = re.search(r"キャリア志向：(.+?)(?:\n・|$)", text, re.S)
    return m.group(1).strip() if m else None


def classify_career(s):
    if s is None:
        return "unknown"
    if "管理職" in s and ("志向" in s or "目指" in s):
        return "管理職志向"
    if "専門職" in s or "専門性" in s:
        return "専門職志向"
    if "安定" in s:
        return "安定志向"
    return "未定"


train_persona["career"] = train_persona["入社時メモ"].apply(extract_career_section).apply(classify_career)

role_order = {"メンバー": 0, "シニア": 1, "リード": 2, "エキスパート": 3, "シニアエキスパート": 4, "マネージャー": 4}
train_persona["role_rank"] = train_persona["初期役割"].map(role_order)
train_persona["ambitious"] = train_persona["career"].isin(["管理職志向", "専門職志向"])

members = train_persona[train_persona["role_rank"] == 0]
print("メンバー内(role_rank=0) n=", len(members))
ct = pd.crosstab(members["ambitious"], members[TARGET_COL])
chi2, p, dof, exp = chi2_contingency(ct)
print(members[TARGET_COL].groupby(members["ambitious"]).agg(["mean", "count"]))
print(f"p={p:.4f}")

print("\n(参考)初期役割そのものの効果:")
print(train_persona[TARGET_COL].groupby(train_persona["role_rank"]).agg(["mean", "count"]))

メンバー内(role_rank=0) n= 2263
               mean  count
ambitious                 
False      0.556962    632
True       0.520540   1631
p=0.1309

(参考)初期役割そのものの効果:
               mean  count
role_rank                 
0          0.530711   2263
1          0.674740    289
2          0.768212    151
3          0.750000     44
4          1.000000     14


### 考察（5）: 志向×役割ミスマッチは初期役割そのものの効果に吸収される（負の結果）

メンバーに限定すると、志向がambitious(管理職/専門職志向)かどうかで定着率に有意差はない
（p=0.13）。全体で見えた大きな差は、role_rankそのもの（既にモデルに投入済み）の効果に
起因する交絡であり、志向×役割の交互作用に独立した価値はない。**不採用とする。**

## 6. 転居許容×希望勤務地マッチの交互作用（最重要発見）

`data_exploration_v3.ipynb`のブロックE（転居許容フラグ、相関済み）と`data_exploration_v4.ipynb`の
ブロックJ（希望勤務地マッチ度）は、それぞれ単体で有効な特徴量として確認済みだが、
**両者を組み合わせた「転居を望まないのに、望んだ場所と違う場所に配属された」というダブルの
悪条件**は未検証だった。

In [9]:
NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location(s):
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


ws_section = train_persona["入社時メモ"].apply(extract_workstyle_section)
train_persona["reloc_ok"] = ws_section.apply(classify_reloc)
train_persona["desired_loc"] = ws_section.apply(extract_desired_location)

loc_match = train_persona["desired_loc"] == train_persona["初期勤務地"]
valid_loc = train_persona["desired_loc"].notna()
valid_both = valid_loc & train_persona["reloc_ok"].notna()

print(f"希望勤務地 抽出率: {valid_loc.mean():.3f}, 転居許容 抽出率: {train_persona['reloc_ok'].notna().mean():.3f}")
print(f"両方有効なサンプル数: {valid_both.sum()} / {len(train_persona)}")

# J単体の再現確認
print("\n[再確認] J(希望勤務地マッチ度)単体:")
print(y[valid_loc].groupby(loc_match[valid_loc]).agg(["mean", "count"]))

希望勤務地 抽出率: 0.886, 転居許容 抽出率: 0.933
両方有効なサンプル数: 2425 / 2761

[再確認] J(希望勤務地マッチ度)単体:
           mean  count
False  0.368151    584
True   0.620505   1863


In [10]:
combo = train_persona.loc[valid_both, "reloc_ok"].astype(str) + "_match=" + loc_match.loc[valid_both].astype(str)
print("転居許容 × 勤務地マッチ の4群:")
print(y[valid_both].groupby(combo).agg(["mean", "count"]))

ct4 = pd.crosstab(combo, y[valid_both])
chi2, p, dof, exp = chi2_contingency(ct4)
print(f"\n4群カイ二乗検定: chi2={chi2:.2f}, p={p:.2e}")

double_bad = ((train_persona["reloc_ok"] == False) & (~loc_match)).astype(int)
double_bad_valid = double_bad[valid_both]
yv = y[valid_both]

ct2 = pd.crosstab(double_bad_valid, yv)
chi2b, pb, dofb, expb = chi2_contingency(ct2)
odds_ratio = (ct2.loc[1, 1] * ct2.loc[0, 0]) / (ct2.loc[1, 0] * ct2.loc[0, 1])

print("\n[新規特徴量] ダブル悪条件フラグ（転居許容せず AND 勤務地不一致）:")
print(yv.groupby(double_bad_valid).agg(["mean", "count"]))
print(f"カイ二乗検定: chi2={chi2b:.2f}, p={pb:.2e}, オッズ比={odds_ratio:.3f}")

転居許容 × 勤務地マッチ の4群:
                       mean  count
False_match=False  0.230994    342
False_match=True   0.615672   1072
True_match=False   0.557447    235
True_match=True    0.628866    776

4群カイ二乗検定: chi2=178.67, p=1.71e-38

[新規特徴量] ダブル悪条件フラグ（転居許容せず AND 勤務地不一致）:
       mean  count
0  0.614018   2083
1  0.230994    342
カイ二乗検定: chi2=173.36, p=1.37e-39, オッズ比=0.189


In [11]:
# Test側でも同様に計算できるか（欠損率・該当件数の確認）
ws_test = test_persona["入社時メモ"].apply(extract_workstyle_section)
test_reloc = ws_test.apply(classify_reloc)
test_loc = ws_test.apply(extract_desired_location)
test_match = test_loc == test_persona["初期勤務地"]
test_double_bad = (test_reloc == False) & test_loc.notna() & (~test_match)

print(f"Test: reloc_ok欠損率={test_reloc.isna().mean():.3f}, desired_loc欠損率={test_loc.isna().mean():.3f}")
print(f"Test: ダブル悪条件 該当件数 = {test_double_bad.sum()} / {len(test_persona)} ({test_double_bad.mean():.3f})")
print(f"(参考)Train全体: ダブル悪条件 該当件数 = {double_bad_valid.sum()} / {len(train_persona)} ({double_bad_valid.sum() / len(train_persona):.3f})")
print(f"(参考)Train有効サンプル内: ダブル悪条件 該当割合 = {double_bad_valid.mean():.3f}")

Test: reloc_ok欠損率=0.071, desired_loc欠損率=0.149
Test: ダブル悪条件 該当件数 = 272 / 2502 (0.109)
(参考)Train全体: ダブル悪条件 該当件数 = 342 / 2761 (0.124)
(参考)Train有効サンプル内: ダブル悪条件 該当割合 = 0.141


### 考察（6）: E・Jを超える最大級の効果量、かつTestでも再現可能

**「転居を伴う異動を許容しない」かつ「希望と異なる勤務地に配属された」社員（n=342, 12.4%）の
定着率はわずか23.1%**で、それ以外の社員（61.4%）との差は**38.3%pt**、オッズ比0.189（約5.3倍
定着しにくい）。カイ二乗検定 p=1.4×10⁻³⁹は、これまで本プロジェクトで確認された全ての単一
特徴量（J: p=1.4×10⁻²⁶、K: 有意だが不採用等）を上回る有意性であり、**効果量（38.3%pt）も
J単体（25.3%pt）を上回る、本プロジェクト最大の発見**である。

4群の内訳を見ると、転居を許容している社員は勤務地の不一致による影響が比較的小さい
（62.9%→55.7%、-7.2%pt）のに対し、転居を許容していない社員では影響が劇的に大きい
（61.6%→23.1%、-38.5%pt）。これは単純な加法的効果ではなく、**「転居を望まない」という
条件が「配属地の裏切り」の痛みを増幅する交互作用**であり、E・Jそれぞれの成分が既にモデルに
入っていても、決定木が自動的に効率よく学習できるとは限らない（Jの発見時と同じ論拠）。

Testデータでも同じ抽出ロジックが同程度の欠損率・該当割合（12.4%→10.9%）で機能することを
確認済みであり、リークの心配もない（入社時に確定する情報のみを使用）。

**新規特徴量として最優先で実装・検証する価値が高い。**

## 7. 総合考察と次のアクション

| # | 分析 | 判定 | 効果量 |
|---|---|---|---|
| 1 | 部署IDの職種超過情報 | **不採用(null)** | 職種内偏差の相関 ≈ -0.003 |
| 2 | 学歴×年齢整合性 | **不採用(null)** | 相関 0.004（新卒）/ -0.052（中途） |
| 3 | 初任給の勤務地内偏差 | **不採用(redundant)** | 初任給そのものと重複、両コントロール後は -0.022 |
| 4 | 前職職種×初期職種マッチ | **不採用(null)** | 68.6% vs 68.7%、p=1.0 |
| 5 | キャリア志向×役割トラック | **不採用(redundant)** | 初期役割の交絡、メンバー内ではp=0.13 |
| 6 | 転居許容×勤務地マッチ交互作用 | **最有望** | 差38.3%pt, p=1.4×10⁻³⁹（本プロジェクト最大） |

### 次のアクション

1. **6の「ダブル悪条件フラグ」（ブロックL想定）を最優先で実装**し、`27_`想定のノートブックで
   `18_`のベースライン（Eなし・Jなし）に対して単体アブレーションを行う。
   `E_memo`が確立した「E_memoに追加ではなく18_ベースラインへの単体追加」という提出プロトコルを踏襲する。
2. Lが18_ベースラインに対して健全に汎化する場合、E・J・Lを組み合わせた場合の効果も検証対象とする
   （ただしcombo_EFG・combo_EG・combo_EIの教訓を踏まえ、複数ブロック合成は検証で改善が
   出ても即採用せず、必ずPublicで確認する）。
3. 1〜5の否定的結果（部署ID・学歴年齢整合性・勤務地内偏差・前職職種マッチ・志向トラック整合性）は
   いずれも「入社時固定」系候補としては筋が良さそうに見えて実際は効果がなかった事例であり、
   今後の探索でも「既存の強い特徴量（初期職種・初期役割・初任給等）に吸収される交絡」に
   注意する必要がある。
